In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

class RiskFeatureSelector:
    def __init__(self, target_col, iv_threshold=0.02, corr_threshold=0.85):
        self.target_col = target_col
        self.iv_threshold = iv_threshold
        self.corr_threshold = corr_threshold
        self.iv_dict = {}

    def calculate_iv(self, df, feature):
        """Tự tính Information Value (IV) và Weight of Evidence (WOE)"""
        # Tách riêng xử lý biến liên tục và biến phân loại
        if df[feature].dtype in ['float64', 'float32', 'int64', 'int32', 'int8']:
            # Chia biến liên tục thành 10 nhóm (bins) có số lượng phần tử xấp xỉ nhau
            # duplicates='drop' để xử lý các cột có nhiều giá trị 0 trùng lặp
            bins = pd.qcut(df[feature], q=10, duplicates='drop')
        else:
            # Biến phân loại (category/object) thì giữ nguyên nhóm
            bins = df[feature]

        # Đếm số lượng Tốt (0) và Xấu (1) trong từng nhóm
        grouped = df.groupby(bins, observed=False)[self.target_col].agg(['count', 'sum'])
        grouped.columns = ['Total', 'Bad']
        grouped['Good'] = grouped['Total'] - grouped['Bad']

        # Xử lý lỗi chia cho 0 bằng cách cộng thêm một số cực nhỏ (epsilon)
        grouped['Bad'] = np.where(grouped['Bad'] == 0, 0.0001, grouped['Bad'])
        grouped['Good'] = np.where(grouped['Good'] == 0, 0.0001, grouped['Good'])

        # Tính tổng số Tốt và Xấu của toàn bộ tập dữ liệu
        total_bad = df[self.target_col].sum()
        total_good = df.shape[0] - total_bad

        # Tính tỷ lệ % và tính WOE, IV cho từng nhóm
        grouped['Bad_Rate'] = grouped['Bad'] / total_bad
        grouped['Good_Rate'] = grouped['Good'] / total_good
        grouped['WOE'] = np.log(grouped['Good_Rate'] / grouped['Bad_Rate'])
        grouped['IV'] = (grouped['Good_Rate'] - grouped['Bad_Rate']) * grouped['WOE']

        # Trả về tổng IV của đặc trưng này
        return grouped['IV'].sum()

    def fit_transform(self, df):
        print("BƯỚC 1.1: Loại bỏ Metadata phi nghiệp vụ...")
        cols_to_drop = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM']
        features = [c for c in df.columns if c not in cols_to_drop and c != self.target_col]

        print(f"BƯỚC 1.2: Bắt đầu tính IV cho {len(features)} biến...")
        selected_by_iv = []
        for col in features:
            iv = self.calculate_iv(df, col)
            self.iv_dict[col] = iv

            # Chỉ giữ lại biến có IV >= 0.02 (Sức mạnh từ mức yếu trở lên)
            if iv >= self.iv_threshold:
                selected_by_iv.append(col)

        print(f" -> Đã loại bỏ {len(features) - len(selected_by_iv)} biến có IV < {self.iv_threshold}")
        print(f" -> Số biến giữ lại sau khi lọc IV: {len(selected_by_iv)}")

        print("\nBƯỚC 1.3: Lọc đa cộng tuyến (Giữ biến có IV cao hơn)...")
        # Chỉ tính tương quan trên các biến liên tục
        num_cols = df[selected_by_iv].select_dtypes(include=[np.number]).columns.tolist()
        corr_matrix = df[num_cols].corr(method='spearman').abs()

        # Lấy nửa trên của ma trận tương quan để tránh tính lặp
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        # Tìm các cặp có độ tương quan > 0.85
        to_drop_by_corr = set()
        for col in upper.columns:
            highly_correlated_with = upper[col][upper[col] > self.corr_threshold].index.tolist()
            for correlated_col in highly_correlated_with:
                # Trọng tài: So sánh IV. Ai thấp hơn thì bị xóa!
                if self.iv_dict[col] > self.iv_dict[correlated_col]:
                    to_drop_by_corr.add(correlated_col)
                else:
                    to_drop_by_corr.add(col)

        final_features = [c for c in selected_by_iv if c not in to_drop_by_corr]

        print(f" -> Đã tìm thấy và loại bỏ {len(to_drop_by_corr)} biến bị đa cộng tuyến.")
        print(f" -> KẾT QUẢ CUỐI CÙNG BƯỚC 1: Còn lại {len(final_features)} biến vàng!")

        # Trả về DataFrame chỉ chứa Target và các biến Vàng
        return df[[self.target_col] + final_features], self.iv_dict, final_features

# # ==========================================
# # CÁCH CHẠY CODE (Giả sử df của bạn đã được load)
# # ==========================================
# df = pd.read_parquet("D:\XAI_risk_score\data\credit_data_processed_v2.parquet") # Thay bằng cách bạn load data
#
# # Chú ý quan trọng: Trong thực tế, để tránh Data Leakage,
# # Bạn PHẢI CHIA TẬP TRAIN/TEST TRƯỚC, rồi mới chạy fit_transform trên tập TRAIN.
#
# selector = RiskFeatureSelector(target_col='target', iv_threshold=0.02, corr_threshold=0.85)
#
# # df_clean chính là tập dữ liệu đã dọn sạch rác, sẵn sàng đưa vào XGBoost
# df_clean, iv_scores, final_cols = selector.fit_transform(df)
#
# # In thử Top 10 biến mạnh nhất theo IV
# print("\n🔥 TOP 10 BIẾN MẠNH NHẤT THEO IV:")
# iv_df = pd.DataFrame(list(iv_scores.items()), columns=['Feature', 'IV']).sort_values(by='IV', ascending=False)
# print(iv_df[iv_df['Feature'].isin(final_cols)].head(10))

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load dữ liệu
df = pd.read_parquet(r"/data/processed/credit_data_processed_v2.parquet")

print("--- BẮT ĐẦU CHIA TẬP DỮ LIỆU ---")
# 2. CHIA TẬP TRAIN/TEST NGAY LẬP TỨC (Dùng stratify để giữ nguyên tỷ lệ mất cân bằng của target)
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['target'] # Cực kỳ quan trọng với dữ liệu mất cân bằng
)

print(f"Kích thước tập Train: {df_train.shape}")
print(f"Kích thước tập Test: {df_test.shape}")

# 3. KHỞI TẠO VÀ CHẠY FEATURE SELECTION CHỈ TRÊN TẬP TRAIN
print("\n--- BẮT ĐẦU FEATURE SELECTION TRÊN TẬP TRAIN ---")
selector = RiskFeatureSelector(target_col='target', iv_threshold=0.02, corr_threshold=0.85)

# fit_transform SẼ HỌC logic chọn cột và trả ra tập Train sạch
df_train_clean, iv_scores, final_cols = selector.fit_transform(df_train)

# 4. ÁP DỤNG KẾT QUẢ CHO TẬP TEST (Chỉ lấy những cột mà tập Train đã chọn)
# Tuyệt đối không chạy lại fit_transform trên tập Test
print("\n--- ĐỒNG BỘ CẤU TRÚC CHO TẬP TEST ---")
df_test_clean = df_test[['target'] + final_cols]

print(f"Kích thước tập Train sạch: {df_train_clean.shape}")
print(f"Kích thước tập Test sạch: {df_test_clean.shape}")

# In thử Top 10 biến mạnh nhất theo IV
print("\n🔥 TOP 10 BIẾN MẠNH NHẤT THEO IV:")
iv_df = pd.DataFrame(list(iv_scores.items()), columns=['Feature', 'IV']).sort_values(by='IV', ascending=False)
print(iv_df[iv_df['Feature'].isin(final_cols)].head(10))

--- BẮT ĐẦU CHIA TẬP DỮ LIỆU ---
Kích thước tập Train: (418076, 49)
Kích thước tập Test: (104520, 49)

--- BẮT ĐẦU FEATURE SELECTION TRÊN TẬP TRAIN ---
BƯỚC 1.1: Loại bỏ Metadata phi nghiệp vụ...
BƯỚC 1.2: Bắt đầu tính IV cho 44 biến...
 -> Đã loại bỏ 11 biến có IV < 0.02
 -> Số biến giữ lại sau khi lọc IV: 33

BƯỚC 1.3: Lọc đa cộng tuyến (Giữ biến có IV cao hơn)...
 -> Đã tìm thấy và loại bỏ 5 biến bị đa cộng tuyến.
 -> KẾT QUẢ CUỐI CÙNG BƯỚC 1: Còn lại 28 biến vàng!

--- ĐỒNG BỘ CẤU TRÚC CHO TẬP TEST ---
Kích thước tập Train sạch: (418076, 29)
Kích thước tập Test sạch: (104520, 29)

🔥 TOP 10 BIẾN MẠNH NHẤT THEO IV:
                         Feature        IV
20     avgdbddpdlast24m_3658932P  0.255989
23  maxdbddpdtollast12m_3658940P  0.218951
2             maxdpdlast24m_143P  0.214234
7    pctinstlsallpaidlat10d_839L  0.213438
1             maxdpdlast12m_727P  0.177987
0     maxdpdfrom6mto36m_3546853P  0.176430
28                  days180_256L  0.166404
24   maxdbddpdtollast6m_4187119

# 📝 BƯỚC 2: LỌC TINH ĐẶC TRƯNG BẰNG PHƯƠNG PHÁP NHÚNG (EMBEDDED METHODS)

## 1. Tổng quan
Sau khi hoàn tất lọc thô bằng toán học thống kê ở Bước 1 (Filter Methods), hệ thống chuyển sang **Bước 2: Phương pháp Nhúng**. Mục tiêu của bước này là đánh giá sức mạnh thực chiến của các đặc trưng thông qua lăng kính của một mô hình học máy thực thụ.

Thay vì thêm/bớt biến một cách thủ công, Phương pháp Nhúng lồng ghép việc đánh giá đặc trưng vào ngay bên trong quá trình huấn luyện thuật toán, giúp tìm ra tập con đặc trưng tối ưu với chi phí tính toán thấp nhất.

## 2. Kỹ thuật cốt lõi: Tăng cường Gradient (LightGBM)
Dự án sử dụng **Shallow LightGBM** (Cây quyết định nông kết hợp Gradient Boosting) làm "mô hình giám khảo" vì các ưu điểm vượt trội:
* **Tích hợp đồng thời:** Thuật toán tự động tính toán điểm **Information Gain** (Độ tăng thông tin) cho từng biến mỗi khi chẻ nhánh.
* **Chống rò rỉ và Overfitting:** Cấu hình cây nông (`max_depth=3`) ép mô hình chỉ học các tín hiệu lõi mạnh nhất, kết hợp cơ chế dừng sớm (Early Stopping) trên tập Validation.
* **Tối ưu Pipeline:** Tự động xử lý giá trị khuyết thiếu (`NaN`) và không yêu cầu chuẩn hóa dữ liệu (Scaling).

## 3. Tiêu chuẩn đánh giá (Sanity Check)
Bảng xếp hạng đặc trưng chỉ được tin cậy khi mô hình giám khảo chứng minh được năng lực phân loại. Hệ thống sử dụng 4 nhóm chỉ số kiểm định khắt khe:

### A. Nhóm Xếp hạng (Ranking Metrics)
Đo lường năng lực phân tách rủi ro giữa khách hàng Tốt và Xấu.
* **ROC-AUC:** Kỳ vọng mức > 0.70.
* **Hệ số Gini:** Tiêu chuẩn vàng trong rủi ro tín dụng. Kỳ vọng mức > 0.40.
  $$\text{Gini} = 2 \times \text{AUC} - 1$$

### B. Nhóm Dữ liệu Mất cân bằng (Imbalance Metrics)
Vì tỷ lệ nợ xấu rất thấp (khoảng 3%), các chỉ số Accuracy bị loại bỏ, thay thế bằng:
* **PR-AUC (Average Precision):** Đánh giá độ chuẩn xác trên tập thiểu số. Kỳ vọng 0.15 - 0.25.

### C. Nhóm Nghiệp vụ Ngân hàng
* **KS Statistic (Kolmogorov-Smirnov):** Đo lường khoảng cách lớn nhất giữa hai phân phối tích lũy của khách hàng Tốt và Xấu. Kỳ vọng đạt 0.30 - 0.50.

## 4. Cơ chế thanh lọc
Kết thúc quá trình huấn luyện, hệ thống trích xuất bảng xếp hạng `feature_importances_` (dựa trên mức độ đóng góp `Gain`).
* **Quy tắc cắt tỉa:** Các đặc trưng có `Gain = 0` (mô hình không sử dụng để phân loại dù chỉ một lần) sẽ bị loại bỏ hoàn toàn khỏi đường ống dữ liệu (Data Pipeline).
* **Đầu ra:** Một tập dữ liệu siêu sạch và tinh gọn, sẵn sàng cho việc huấn luyện mô hình sâu (Deep Models) ở giai đoạn cuối.

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp

print("\n=======================================================")
print("🚀 BẮT ĐẦU BƯỚC 2: LỌC TINH BẰNG MÔ HÌNH (EMBEDDED METHODS)")
print("=======================================================\n")

# 1. TÁCH TẬP VALIDATION TỪ TẬP TRAIN SẠCH (CỦA BƯỚC 1)
# Nhắc lại: df_train_clean là kết quả thu được từ Bước 1
X = df_train_clean.drop(columns=['target'])
y = df_train_clean['target']

X_train_s2, X_val_s2, y_train_s2, y_val_s2 = train_test_split(
    X, y,
    test_size=0.2,       # Lấy 20% làm Validation
    random_state=42,
    stratify=y           # Giữ nguyên tỷ lệ nợ xấu 3.3%
)

print(f"Kích thước tập Huấn luyện (Bước 2): {X_train_s2.shape}")
print(f"Kích thước tập Validation (Bước 2): {X_val_s2.shape}")

# 2. KHỞI TẠO MÔ HÌNH NHẸ (SHALLOW TREE)
clf_light = lgb.LGBMClassifier(
    n_estimators=400,        # Cho phép học tối đa 200 cây
    max_depth=3,             # Cây siêu nông -> Ép mô hình chỉ học các biến cốt lõi
    learning_rate=0.05,
    class_weight='balanced', # Xử lý mất cân bằng dữ liệu
    importance_type='gain',  # Tính quan trọng theo mức độ đóng góp (Gain)
    random_state=42,
    n_jobs=-1
)

# 3. HUẤN LUYỆN VỚI EARLY STOPPING
print("\nĐang huấn luyện LightGBM...")
clf_light.fit(
    X_train_s2, y_train_s2,
    eval_set=[(X_val_s2, y_val_s2)],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20, verbose=False),
        lgb.log_evaluation(period=20) # In log mỗi 20 vòng
    ]
)

# Lấy iteration tốt nhất
best_iter = clf_light.best_iteration_
print(f"-> Mô hình dừng sớm tại cây thứ: {best_iter}")

# 4. ĐÁNH GIÁ SANITY CHECK BẰNG CÁC CHỈ SỐ NGHIỆP VỤ NGÂN HÀNG
# Lấy xác suất dự đoán (probability) trên tập Validation
y_val_prob = clf_light.predict_proba(X_val_s2)[:, 1]

# Tính toán
auc = roc_auc_score(y_val_s2, y_val_prob)
gini = 2 * auc - 1
pr_auc = average_precision_score(y_val_s2, y_val_prob)
# KS Statistic: Khoảng cách phân phối giữa người Tốt và Xấu
ks_stat, _ = ks_2samp(y_val_prob[y_val_s2 == 1], y_val_prob[y_val_s2 == 0])

print("\n--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VALIDATION ---")
print(f"🎯 ROC-AUC      : {auc:.4f} (Kỳ vọng > 0.70)")
print(f"🎯 Gini         : {gini:.4f} (Kỳ vọng > 0.40)")
print(f"🎯 PR-AUC       : {pr_auc:.4f} (Do nợ xấu cực thấp 3.3%, kỳ vọng 0.15 - 0.25)")
print(f"🎯 KS Statistic : {ks_stat:.4f} (Kỳ vọng 0.30 - 0.50)")

# 5. TRÍCH XUẤT FEATURE IMPORTANCE VÀ CHỐT SỔ ĐẶC TRƯNG
feature_imp = pd.DataFrame({
    'Feature': X_train_s2.columns,
    'Gain': clf_light.feature_importances_
}).sort_values(by='Gain', ascending=False)

# TIÊU CHÍ LOẠI BỎ: Xóa bỏ các biến mà mô hình không thèm dùng đến (Gain = 0)
# Hoặc chỉ lấy top 30-40 biến nếu hệ thống vẫn còn quá nhiều biến
final_features_step2 = feature_imp[feature_imp['Gain'] > 0]['Feature'].tolist()

print(f"\n--- KẾT QUẢ CUỐI CÙNG BƯỚC 2 ---")
print(f"Số lượng đặc trưng đưa vào: {X_train_s2.shape[1]}")
print(f"Số lượng đặc trưng được giữ lại: {len(final_features_step2)}")

print("\n🔥 TOP 10 ĐẶC TRƯNG CỐT LÕI (XẾP THEO GAIN):")
print(feature_imp.head(10))

# 6. ĐỒNG BỘ CẤU TRÚC CHO TẬP TRAIN & TEST ĐỂ SẴN SÀNG CHO PHASE MODELING CUỐI
# (Tập test gốc ban đầu ở Bước 1)
df_train_final = df_train_clean[['target'] + final_features_step2]
df_test_final = df_test_clean[['target'] + final_features_step2]

print("\n✅ Phễu lọc hoàn tất! Dữ liệu đã sẵn sàng để huấn luyện mô hình sâu (Deep Model) và giải thích SHAP.")


🚀 BẮT ĐẦU BƯỚC 2: LỌC TINH BẰNG MÔ HÌNH (EMBEDDED METHODS)

Kích thước tập Huấn luyện (Bước 2): (334460, 28)
Kích thước tập Validation (Bước 2): (83616, 28)

Đang huấn luyện LightGBM...
[LightGBM] [Info] Number of positive: 11023, number of negative: 323437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014177 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3910
[LightGBM] [Info] Number of data points in the train set: 334460, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt

In [7]:
import json
import os

# Tạo thư mục lưu model và config nếu chưa có
os.makedirs("../model_artifacts", exist_ok=True)

# 1. Lưu tập dữ liệu Vàng (Gold Dataset)
print("Đang lưu tập dữ liệu Vàng...")
df_train_final.to_parquet("model_artifacts/train_gold.parquet", index=False)
df_test_final.to_parquet("model_artifacts/test_gold.parquet", index=False)

# 2. Lưu danh sách 28 Feature ra file JSON để sau này API đọc
with open("../model_artifacts/selected_features.json", "w") as f:
    json.dump(final_features_step2, f)

print("✅ Đã lưu xong dữ liệu và cấu hình!")

Đang lưu tập dữ liệu Vàng...
✅ Đã lưu xong dữ liệu và cấu hình!


In [8]:
import pandas as pd
import json
import os

# Tạo thư mục lưu model và config nếu chưa có
os.makedirs("../model_artifacts", exist_ok=True)

# 1. TÁI TẠO DATAFRAME CHO TRAIN VÀ VAL (Gộp X và y lại)
# Chỉ lấy đúng 28 feature đã chốt (final_features_step2)
df_train_gold = X_train_s2[final_features_step2].copy()
df_train_gold['target'] = y_train_s2

df_val_gold = X_val_s2[final_features_step2].copy()
df_val_gold['target'] = y_val_s2

# Tập Test thì lấy trực tiếp từ df_test_clean ở Bước 1
df_test_gold = df_test_clean[['target'] + final_features_step2].copy()

# 2. LƯU CẢ 3 TẬP DỮ LIỆU VÀNG
print("Đang lưu tập dữ liệu Vàng (Train, Val, Test)...")
df_train_gold.to_parquet("model_artifacts/train_gold.parquet", index=False)
df_val_gold.to_parquet("model_artifacts/val_gold.parquet", index=False)
df_test_gold.to_parquet("model_artifacts/test_gold.parquet", index=False)

# 3. LƯU DANH SÁCH 28 FEATURE
with open("../model_artifacts/selected_features.json", "w") as f:
    json.dump(final_features_step2, f)

print(f"✅ Đã lưu xong! Kích thước Train: {df_train_gold.shape}, Val: {df_val_gold.shape}, Test: {df_test_gold.shape}")

Đang lưu tập dữ liệu Vàng (Train, Val, Test)...
✅ Đã lưu xong! Kích thước Train: (334460, 29), Val: (83616, 29), Test: (104520, 29)
